# 예제 02. `Conversation` 클래스 만들기 · Example 02 — Building the `Conversation` Class

전산실험 2 · AI Agent 만들기 · **4주차** · 예상 소요 35분
Computational Experiments 2 · Building AI Agents · **Week 4** · about 35 minutes

## 목표 · Objectives
- 대화 기록을 클래스로 감싸 예산 관리를 안쪽에 숨긴다
- **무엇을 보존하고 무엇을 버릴지** 우선순위를 정한다
- 불변식("대화는 항상 예산 안에 있다")을 설계에 넣는다
- 대화를 저장하고 복원한다

> **EN**
> - Wrap the conversation history in a class so budget management lives inside it
> - Decide the priority order of **what to keep and what to discard**
> - Build an invariant into the design: "the conversation always fits the budget"
> - Save a conversation and restore it

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

EN — Shared LLM wrapper for "Computational Experiments 2: Building AI Agents".
Calls the Keimyung Statistics department server and Google Gemini the same way.
Both expose an OpenAI-compatible API, so we use the single openai SDK and only
swap `base_url` and `model`.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기

 EN — This repository is PUBLIC, so never write the department server URL or
 the API key in code. Both are announced in class; store them in Colab Secrets.
     Key icon (🔑) on the left → new secret → turn on notebook access
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예 · Example:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
                                         # Check the server is alive first
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로 / Fallback route
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정 / Server configuration
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
# EN — The department server URL is never written here; it is read from the
#      Colab secret KMU_BASE_URL. The model tag is harmless to publish, so it
#      stays a constant.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
# EN — Fallback route. Get a free key from Google AI Studio. (public URL)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다 / read from Secrets at runtime
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기 / Loading secrets
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    EN — Look up a configuration value in a safe order:
    1) Colab Secrets (key icon on the left)  2) environment variable  3) prompt.
    The point is to keep the value out of the source. The moment a notebook or
    module is pushed to GitHub, anything written in the code is world-readable —
    the internal server URL just as much as the API key.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
             EN — True hides the typed value (getpass). Pass False for values
             like a URL where you need to see typos.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수 / environment variable
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력 / type it in
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다 / ask only once per runtime
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)

    EN — Fetch the API key. Alias of get_secret, kept under the week-1 name.
    """
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.

    EN — Return the server URL with any trailing '/' removed. The department
    URL is never stored in the repository, so it comes from Secrets. If you
    saved the full '/chat/completions' path, it is trimmed for you.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    # EN — The openai SDK appends /chat/completions itself, so strip it here in
    #      case the whole path was saved as the secret.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용 / reuse within the session
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트 / LLM client
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    EN — A thin client wrapping any OpenAI-compatible server.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
        EN — Which server to use. Defaults to the department server.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
        EN — Override the defaults. You normally leave these alone.
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        # EN — Never print the URL; screen shares and screenshots leak it easily.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 / Core calls -------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용.

        EN — Return the full response object, so you can inspect `usage`
        (token counts).
        """
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다.

        EN — Send the message list and return only the response text.
        """
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과.

        EN — Yield the response chunk by chunk, giving the typing effect.
        """
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 / Convenience methods ---------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다.

        EN — A one-shot question, for when building `messages` by hand is
        tedious.
        """
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.

        EN — Retry transient network/server errors, so that a lab with many
        calls (a 30-repetition experiment, say) does not die on one failure.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반 / any network or server error
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다 / back off progressively
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다.

        EN — Ask the server how many input tokens the messages use; the output
        is capped at 1 token to avoid waste.
        """
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수 / Module-level convenience functions
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.

    EN — The simplest one-line call. Internally it builds one LLM object and
    reuses it.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단 / Server diagnostics
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    EN — Check that the server is alive and see which models it serves. This is
    the very first lab of the course; on failure it prints what to check.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    # EN — Never print the full URL; show the host masked.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다.

    EN — Mask the host so screen shares and screenshots do not leak it.
    """
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들. / What to check when the connection fails."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile memory.py
"""전산실험2 4주차 : 대화 상태와 메모리.

모델은 이전 대화를 기억하지 못한다. 기억은 **우리 코드가** 만든다.
그런데 대화를 통째로 다시 보내다 보면 컨텍스트 윈도우에 부딪힌다.

이 모듈은 세 가지를 제공한다.

    Conversation   대화 기록을 들고 있는 그릇 (system 은 항상 보존)
    예산 관리       토큰을 세고, 넘치면 오래된 턴을 잘라낸다
    요약 압축       잘라내는 대신 요약으로 갈아끼운다 (정보 손실 최소화)

EN — Week 4: conversation state and memory.
The model remembers nothing; OUR code creates the memory. But resending the
whole conversation eventually hits the context window. This module offers:
Conversation (a container for the history, with the system message always
preserved), budget management (count tokens and drop old turns when over), and
summary compression (replace old turns with a summary instead of dropping them).

사용 예 · Example:
    from kmu_llm import LLM
    from memory import Conversation

    llm = LLM()
    conv = Conversation(llm, system="너는 통계 조교다.", max_tokens=2000)
    print(conv.say("표준편차가 뭔가요?"))
    print(conv.say("그럼 표준오차는요?"))     # 앞 대화를 기억한다 / remembers
    conv.summary()                          # 현재 토큰 사용 현황 / token usage
"""

from __future__ import annotations

from typing import Any, Literal

__all__ = ["estimate_tokens", "Message", "Conversation"]

Role = Literal["system", "user", "assistant"]


# --------------------------------------------------------------------------
# 토큰 추정 / Token estimation
# --------------------------------------------------------------------------
def estimate_tokens(text: str) -> int:
    """토큰 수를 대략 추정한다. 서버에 묻지 않고 빠르게 계산한다.

    정확한 값은 모델의 토크나이저에 따라 다르다. 여기서는 경험적 근사를 쓴다.

        한글 1글자  ~ 1.2 토큰   (자소 단위로 잘게 쪼개짐)
        영문 1글자  ~ 0.3 토큰   (단어가 통째로 1토큰이 되는 경우가 많음)

    루프를 돌 때마다 서버에 물어보면 호출이 배로 늘어난다. 예산 관리는
    **근사로 충분**하고, 정확한 값이 필요하면 `LLM.count_tokens()` 를 쓴다.

    EN — Roughly estimate the token count locally, without asking the server.
    The exact number depends on the model's tokenizer; these are empirical
    approximations (one Hangul character ≈ 1.2 tokens because it splits into
    jamo; one Latin character ≈ 0.3 tokens because whole words often become one
    token). Asking the server on every loop iteration would double the number of
    calls — an approximation is good enough for budgeting, and `LLM.count_tokens()`
    is there when you need the exact figure.
    """
    if not text:
        return 0
    hangul = sum(1 for ch in text if "가" <= ch <= "힣")
    other = len(text) - hangul
    return int(hangul * 1.2 + other * 0.3) + 1


class Message(dict):
    """{"role": ..., "content": ...} 에 토큰 수를 얹은 것.

    EN — A {"role": ..., "content": ...} dict with a token count attached.
    """

    def __init__(self, role: Role, content: str) -> None:
        super().__init__(role=role, content=content)

    @property
    def tokens(self) -> int:
        return estimate_tokens(self["content"]) + 4  # 역할 표시 등 부대 비용 / overhead for the role marker

    def as_api(self) -> dict[str, str]:
        return {"role": self["role"], "content": self["content"]}


# --------------------------------------------------------------------------
# 대화 / Conversation
# --------------------------------------------------------------------------
class Conversation:
    """대화 기록을 관리하고 예산을 넘지 않게 유지한다.

    EN — Keeps the conversation history and holds it within the token budget.

    Parameters
    ----------
    llm : LLM 인스턴스 / an LLM instance
    system : 대화 전체에 적용되는 지시. **절대 잘려나가지 않는다.**
             EN — the instruction for the whole conversation; never truncated.
    max_tokens : 전체 대화의 토큰 예산 (응답 공간은 별도)
                 EN — token budget for the conversation (reply space is separate)
    strategy :
        "drop"    오래된 턴을 그냥 버린다 (빠르고 단순, 정보 손실)
                  EN — discard old turns (fast and simple, loses information)
        "summary" 오래된 턴을 요약으로 갈아끼운다 (호출 1회 추가, 손실 적음)
                  EN — replace old turns with a summary (one extra call, less loss)
        "none"    아무것도 하지 않는다 (한계를 직접 보고 싶을 때)
                  EN — do nothing, so you can watch the limit being hit
    keep_recent : 최근 몇 개의 메시지는 무조건 원본으로 보존할지
                  EN — how many recent messages are always kept verbatim
    """

    def __init__(
        self,
        llm: Any,
        system: str | None = None,
        *,
        max_tokens: int = 3000,
        strategy: str = "summary",
        keep_recent: int = 4,
    ) -> None:
        if strategy not in ("drop", "summary", "none"):
            raise ValueError(f"알 수 없는 strategy: {strategy!r}")

        self.llm = llm
        self.max_tokens = max_tokens
        self.strategy = strategy
        self.keep_recent = keep_recent

        self.system = Message("system", system) if system else None
        self.history: list[Message] = []
        self.compressions = 0          # 몇 번 압축했는지 / how many compressions so far
        self.dropped = 0               # 몇 개를 버렸는지 / how many messages were dropped

    # -- 상태 확인 / Inspecting the state ----------------------------------
    def messages(self) -> list[dict[str, str]]:
        """API 로 보낼 형태. / The shape that goes to the API."""
        head = [self.system] if self.system else []
        return [m.as_api() for m in head + self.history]

    @property
    def tokens(self) -> int:
        head = self.system.tokens if self.system else 0
        return head + sum(m.tokens for m in self.history)

    def summary(self) -> dict[str, Any]:
        """현재 상태를 한눈에. / The current state at a glance."""
        return {
            "메시지수": len(self.history) + (1 if self.system else 0),
            "추정토큰": self.tokens,
            "예산": self.max_tokens,
            "사용률": round(self.tokens / self.max_tokens, 3),
            "압축횟수": self.compressions,
            "버린메시지": self.dropped,
        }

    # -- 대화 / Talking ----------------------------------------------------
    def add(self, role: Role, content: str) -> None:
        self.history.append(Message(role, content))

    def say(self, text: str, **kw: Any) -> str:
        """사용자 발화를 넣고 모델의 답을 받아 기록에 남긴다.

        예산은 **보내기 전과 받은 뒤 두 번** 맞춘다. 그래야
        "대화는 항상 예산 안에 있다"는 불변식이 유지되어,
        지금 상태를 보고 다음 호출이 들어갈지 예측할 수 있다.

        EN — Append the user's utterance, get the model's reply, and record it.
        The budget is enforced twice — before sending and after receiving — so
        the invariant "the conversation always fits the budget" holds, which
        lets you predict from the current state whether the next call will fit.
        """
        self.add("user", text)
        self.fit()                                  # 보내기 전 / before sending
        reply = self.llm.chat(self.messages(), **kw)
        self.add("assistant", reply)
        self.fit()                                  # 받은 뒤 / after receiving
        return reply

    # -- 예산 관리 / Budget management -------------------------------------
    def fit(self) -> None:
        """예산을 넘으면 전략에 따라 줄인다.

        EN — Shrink the history according to the strategy when over budget.
        """
        if self.strategy == "none" or self.tokens <= self.max_tokens:
            return
        if self.strategy == "drop":
            self._drop_old()
        else:
            self._summarize_old()

    def _split(self) -> tuple[list[Message], list[Message]]:
        """(압축 대상인 오래된 것, 보존할 최근 것)

        EN — (the old part to compress, the recent part to keep)
        """
        if len(self.history) <= self.keep_recent:
            return [], self.history
        return self.history[: -self.keep_recent], self.history[-self.keep_recent :]

    def _drop_old(self) -> None:
        """가장 오래된 것부터 예산에 맞을 때까지 버린다.

        `keep_recent` 는 보존하려 하지만, 그것만으로도 예산을 넘으면
        보존 약속을 깨고 더 버린다. 마지막 메시지 하나는 남긴다 —
        그것이 사용자의 방금 질문이기 때문이다.

        EN — Drop from the oldest until the history fits. `keep_recent` is
        honoured where possible, but if even those exceed the budget the promise
        is broken and more is dropped. The last message always survives — it is
        the user's current question.
        """
        old, recent = self._split()
        while old and self.tokens > self.max_tokens:
            old.pop(0)
            self.dropped += 1
            self.history = old + recent
        self.history = old + recent

        # 최근 것만 남았는데도 넘치면, 보존 약속을 깨고 더 버린다
        # EN — if even the recent messages overflow, break the promise and drop more
        while len(self.history) > 1 and self.tokens > self.max_tokens:
            self.history.pop(0)
            self.dropped += 1

        if self.tokens > self.max_tokens:
            # 마지막 메시지 하나가 예산보다 크다. 내용을 몰래 자르지는 않는다.
            # 조용히 틀리느니 시끄럽게 알리는 편이 낫다. (3주차와 같은 원칙)
            # EN — a single message exceeds the budget. We do not silently
            #      truncate it: better to fail loudly than to be quietly wrong
            #      (the same principle as week 3).
            print(
                f"[경고] 마지막 메시지 하나가 예산({self.max_tokens} 토큰)을 "
                f"넘습니다 (현재 {self.tokens}). 예산을 늘리거나 입력을 줄이세요."
            )

    def _summarize_old(self) -> None:
        """오래된 대화를 요약 한 덩어리로 갈아끼운다.

        EN — Replace the old part of the conversation with a single summary.
        """
        old, recent = self._split()
        if not old:
            self._drop_old()          # 줄일 것이 최근 것뿐이면 버리는 수밖에 없다
                                      # EN — if only recent turns remain, dropping is the only option
            return

        기록 = "\n".join(f"[{m['role']}] {m['content']}" for m in old)
        지시 = (
            "다음은 사용자와 조수의 지난 대화다. 이후 대화를 이어가는 데 필요한 "
            "사실만 5문장 이내로 요약하라. 결정된 내용, 사용자가 밝힌 정보, "
            "아직 해결되지 않은 질문을 우선 남겨라. 인사말은 버려라.\n\n"
            f"{기록}"
        )
        try:
            요약 = self.llm.ask(지시, temperature=0.0, max_tokens=400)
        except Exception:
            self._drop_old()          # 요약 호출이 실패하면 버리기로 대체
                                      # EN — if the summary call fails, fall back to dropping
            return

        self.history = [Message("user", f"[이전 대화 요약]\n{요약}")] + recent
        self.compressions += 1

        if self.tokens > self.max_tokens:   # 요약해도 넘치면 결국 버린다 / still over: drop
            self._drop_old()

    # -- 저장 / 복원 · Saving and restoring --------------------------------
    def to_dict(self) -> dict[str, Any]:
        return {
            "system": self.system["content"] if self.system else None,
            "history": [m.as_api() for m in self.history],
            "compressions": self.compressions,
            "dropped": self.dropped,
        }

    @classmethod
    def from_dict(cls, llm: Any, data: dict[str, Any], **kw: Any) -> "Conversation":
        conv = cls(llm, system=data.get("system"), **kw)
        for m in data.get("history", []):
            conv.add(m["role"], m["content"])
        conv.compressions = data.get("compressions", 0)
        conv.dropped = data.get("dropped", 0)
        return conv


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기
# EN — Load the shared module written by the setup cell above.

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요 / switch to "gemini" if the department server is down
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

---
## 1. 우선순위를 먼저 정한다 · Decide the priorities first

무작정 오래된 것부터 버리면 안 됩니다.

| 우선순위 | 무엇 | 왜 |
|---|---|---|
| **절대 보존** | `system` | 사라지면 에이전트가 정체성을 잊는다 |
| **절대 보존** | 사용자의 마지막 질문 | 지금 답해야 할 것 |
| 보존 | 최근 몇 턴 | 대명사·생략된 주어가 앞을 가리킨다 |
| 버려도 됨 | 오래된 인사말 | 정보가 없다 |
| **1순위 폐기** | 실패했다가 고쳐진 시도들 | 3주차 되먹임의 부산물. 해결되면 무용 |

**EN** — Do not simply drop the oldest thing you have.

| Priority | What | Why |
|---|---|---|
| **Never drop** | `system` | Lose it and the agent forgets its identity |
| **Never drop** | The user's last question | It is what has to be answered now |
| Keep | The last few turns | Pronouns and dropped subjects point backwards |
| Droppable | Old pleasantries | They carry no information |
| **Drop first** | Failed attempts that were later fixed | A by-product of the week-3 feedback loop; useless once resolved |

> 3주차에 `system` 과 `user` 를 나눠 두라고 한 이유가 여기서 드러납니다.
> **경계가 명확하면 무엇을 보존할지가 자동으로 정해집니다.**

> **EN** — Here is why week 3 insisted on separating `system` from `user`.
> **When the boundary is clear, what to preserve decides itself.**

---
## 2. 직접 만들어 보기 (축약판) · Build it yourself — a condensed version

전체 구현은 `common/memory.py` 에 있습니다.
먼저 핵심만 30줄로 만들어 봅시다.

> **EN** — The full implementation lives in `common/memory.py`. First, let's build
> the core of it in 30 lines.

In [ ]:
from memory import estimate_tokens

class 내_대화:
    # EN — Our own minimal Conversation class.
    def __init__(self, llm, system=None, max_tokens=1500, keep_recent=4):
        self.llm = llm
        self.system = {"role": "system", "content": system} if system else None
        self.history = []
        self.max_tokens = max_tokens
        self.keep_recent = keep_recent
        self.버린수 = 0

    def messages(self):
        # EN — The message list to send to the API.
        head = [self.system] if self.system else []
        return head + self.history

    @property
    def tokens(self):
        return sum(estimate_tokens(m["content"]) + 4 for m in self.messages())

    def fit(self):
        # system 은 건드리지 않고, 오래된 history 부터 버린다
        # EN — Never touch `system`; drop from the oldest history first.
        오래된 = self.history[:-self.keep_recent] if len(self.history) > self.keep_recent else []
        최근   = self.history[-self.keep_recent:] if len(self.history) > self.keep_recent else self.history
        while 오래된 and self.tokens > self.max_tokens:
            오래된.pop(0)
            self.버린수 += 1
            self.history = 오래된 + 최근
        self.history = 오래된 + 최근

    def say(self, text, **kw):
        self.history.append({"role": "user", "content": text})
        self.fit()                                    # 보내기 전 / before sending
        답 = self.llm.chat(self.messages(), **kw)
        self.history.append({"role": "assistant", "content": 답})
        self.fit()                                    # 받은 뒤 / after receiving
        return 답


c = 내_대화(llm, system="너는 통계 조교다. 2문장 이내로 답한다.", max_tokens=800)
for q in ["표준편차가 뭔가요?", "표준오차와 다른 점은?", "표본이 커지면요?"]:
    print(f"Q: {q}")
    print(f"A: {c.say(q, max_tokens=200)}")
    print(f"   [토큰 {c.tokens}/{c.max_tokens}, 버림 {c.버린수}]\n")

### 설계 결정 하나 짚기 — 왜 `fit()` 을 두 번 부르나 · One design decision — why call `fit()` twice?

보내기 전에만 맞추면, 응답을 받아 붙인 직후에는 **예산을 넘긴 상태**가 됩니다.
그러면 지금 상태를 봐도 다음 호출이 들어갈지 알 수 없습니다.

두 번 맞추면 이런 불변식이 생깁니다.

> **대화는 언제 확인해도 예산 안에 있다.**

불변식이 있으면 디버깅이 훨씬 쉬워집니다.
클래스를 만들 때는 이런 것을 의식적으로 정해 두십시오.

> **EN** — Trim only before sending and, the instant you append the reply, the
> conversation is **over budget** — so inspecting the current state tells you
> nothing about whether the next call will fit. Trimming twice buys the invariant:
> **whenever you look, the conversation fits the budget.** Invariants make
> debugging vastly easier; decide them deliberately when you design a class.

---
## 3. 공식 구현 `memory.Conversation` · The reference implementation

축약판에 몇 가지가 더 있습니다.

| 추가 | 왜 |
|---|---|
| `strategy="summary"` | 버리는 대신 요약으로 압축 (다음 노트북) |
| `strategy="none"` | 한계를 직접 보고 싶을 때 |
| `summary()` | 현재 예산 사용 현황 |
| `to_dict()` / `from_dict()` | 대화 저장·복원 |
| 최근 것만으로도 넘칠 때 처리 | 보존 약속을 깨고 더 버리되 **경고를 낸다** |

**EN** — A few things beyond the condensed version.

| Addition | Why |
|---|---|
| `strategy="summary"` | Compress by summarising instead of dropping (next notebook) |
| `strategy="none"` | For when you want to watch the limit being hit |
| `summary()` | The current budget usage |
| `to_dict()` / `from_dict()` | Save and restore a conversation |
| Handling overflow of the recent turns alone | Break the promise and drop more — **but warn** |

In [ ]:
from memory import Conversation

conv = Conversation(llm, system="너는 통계학 조교다. 2문장 이내로 답한다.",
                    max_tokens=1200, strategy="drop", keep_recent=4)

질문들 = [
    "제 데이터는 환자 120명의 혈압입니다.",
    "치료 전후로 측정했습니다.",
    "어떤 검정을 써야 하나요?",
    "정규성 가정이 깨지면요?",
    "표본이 120명이면 괜찮지 않나요?",
    "유의수준은 0.05로 하겠습니다.",
    "결과 해석은 어떻게 하나요?",
    "효과크기도 봐야 하나요?",
]

for q in 질문들:
    conv.say(q, max_tokens=150)
    s = conv.summary()
    print(f"[{s['메시지수']:2d}개 | {s['추정토큰']:5d}/{s['예산']} | 버림 {s['버린메시지']}]  {q}")

In [ ]:
# 지금 대화에 무엇이 남아 있나? / What is left in the conversation now?
for m in conv.messages():
    print(f"[{m['role']:9s}] {m['content'][:70]}")

`system` 이 맨 위에 그대로 있는지 확인하세요. **절대 잘리지 않습니다.**

그런데 초반의 중요한 정보 — "환자 120명", "치료 전후" — 가 사라졌을 겁니다.
이제 물어보면 어떻게 될까요?

> **EN** — Confirm that `system` is still at the very top: **it is never cut.**
> But the important early facts — "120 patients", "measured before and after" —
> will have vanished. What happens if we ask about them now?

In [ ]:
print(conv.say("제 표본이 몇 명이라고 했죠?", max_tokens=150))

> **정보가 사라졌습니다.** `drop` 전략의 대가입니다.
> 다음 노트북에서 `summary` 전략으로 이 손실을 줄여 봅니다.

> **EN — the information is gone.** That is the price of the `drop` strategy. The
> next notebook reduces the loss with the `summary` strategy.

---
## 4. 저장과 복원 · Saving and restoring

Colab 런타임은 언젠가 끊깁니다. 긴 분석 세션을 이어가려면 저장이 필요합니다.

> **EN** — A Colab runtime disconnects eventually. Continuing a long analysis
> session requires saving it.

In [ ]:
import json

# 저장 / Save
데이터 = conv.to_dict()
with open("conversation.json", "w", encoding="utf-8") as f:
    json.dump(데이터, f, ensure_ascii=False, indent=2)

print("저장된 크기:", len(json.dumps(데이터, ensure_ascii=False)), "글자")
print("메시지 수:", len(데이터["history"]))

In [ ]:
# 복원 / Restore
with open("conversation.json", encoding="utf-8") as f:
    복원데이터 = json.load(f)

conv2 = Conversation.from_dict(llm, 복원데이터,
                               max_tokens=1200, strategy="drop", keep_recent=4)

print("복원 성공:", conv2.messages() == conv.messages())
print()
print(conv2.say("방금 우리가 무슨 이야기를 하고 있었죠?", max_tokens=200))

> `ensure_ascii=False` 를 빼면 한글이 `\uAC00` 같은 이스케이프로 저장됩니다.
> 파일이 커지고 사람이 읽을 수 없게 됩니다.

> **EN** — Omit `ensure_ascii=False` and Korean is stored as escapes like
> `\uAC00`: the file grows and a human can no longer read it.

---
## 연습문제 · Exercises

**문제 1.** `keep_recent` 를 2, 4, 8 로 바꿔 가며 같은 대화를 돌리고,
"표본이 몇 명이라고 했죠?" 에 답할 수 있는지 확인하시오.

**문제 2.** `내_대화` 에 `strategy` 를 추가해 "중요 표시된 메시지는 버리지 않기"를
구현하시오. `say(text, pin=True)` 로 고정할 수 있게 하면 된다.

**문제 3.** 대화가 압축될 때마다 로그를 남기도록 `Conversation` 을 고치시오.
언제 무엇이 버려졌는지 추적할 수 있어야 한다.

**문제 4.** `to_dict()` 는 `llm` 을 저장하지 않는다. 왜 그런가?
저장해야 한다면 무엇이 문제가 되는가?

> **EN — 1.** Run the same conversation with `keep_recent` at 2, 4 and 8, and see
> whether it can still answer "how many subjects did I say I had?"
>
> **EN — 2.** Add a strategy to `내_대화` that never drops messages marked
> important — let `say(text, pin=True)` pin one.
>
> **EN — 3.** Modify `Conversation` to log every compression, so you can trace
> when and what was discarded.
>
> **EN — 4.** `to_dict()` does not save `llm`. Why not? What would go wrong if it
> did?

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 2.** 예시
```python
def say(self, text, pin=False, **kw):
    m = {"role": "user", "content": text, "pin": pin}
    ...
def fit(self):
    버릴수있는 = [m for m in 오래된 if not m.get("pin")]
```
`pin` 은 API 로 보낼 때 빼야 합니다(`role`/`content` 만 허용).
실무 에이전트에서 "사용자가 명시한 제약"을 고정하는 데 자주 쓰는 기법입니다.

**문제 4.** `llm` 은 **API 키를 들고 있습니다.** 대화를 JSON으로 저장하면서
클라이언트까지 직렬화하면 키가 파일에 섞여 들어가고, 그 파일을 커밋하면
2주차에 본 사고가 그대로 재현됩니다.

일반 원칙: **상태(데이터)와 연결(자원)을 분리한다.** 저장하는 것은 대화 내용뿐이고,
복원할 때 `llm` 을 새로 주입합니다. `from_dict(llm, data)` 의 시그니처가
그렇게 생긴 이유입니다.

---

**EN — 2.** Strip `pin` before sending to the API (only `role` and `content` are
allowed). Production agents use this often, to pin down "a constraint the user
stated explicitly".

**EN — 4.** `llm` **holds the API key.** Serialising the client alongside the
conversation would bake the key into the JSON file, and committing that file
reproduces exactly the accident from week 2.

The general principle: **separate state (data) from connections (resources).**
What gets saved is the conversation content alone, and `llm` is injected afresh on
restore — which is why `from_dict(llm, data)` has the signature it has.

</details>

---
## 정리 · Summary

- 예산 관리를 **클래스 안쪽에 숨긴다.** 쓰는 쪽은 `say()` 만 안다
- 보존 우선순위: `system` > 마지막 질문 > 최근 턴 > 나머지
- 불변식을 정해 둔다: **대화는 항상 예산 안에 있다**
- `drop` 은 정보를 영영 잃는다
- 저장할 때 **키를 들고 있는 객체는 직렬화하지 않는다**

> **EN**
> - **Hide budget management inside the class** — the caller only knows `say()`
> - Preservation order: `system` > the last question > recent turns > the rest
> - Fix an invariant: **the conversation always fits the budget**
> - `drop` loses information for good
> - When saving, **never serialise an object that holds a key**

**다음 노트북** → [`03_compression.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week04/03_compression.ipynb)
버리는 대신 요약해서 정보를 지켜 봅니다.